# NB06 — Faithfulness (AOPC + gaze alignment) + Error Taxonomy

**Prerequisite:** NB05 refit (`checkpoints/stage2_refit_best.pth`). Detection (NB04) is frozen.

## Research role
1. **Decision faithfulness (primary):** per-image **AOPC_AUC** with bootstrap CI — does last-attn mark regions the classifier uses?
2. **Gaze alignment (primary companion):** **NSS** + **CC** vs radiologist gaze maps (bootstrap CI). Optional: same metrics on best λ=0 ablation fold.
3. **Error taxonomy (secondary):** FP-A / FP-B(adapted) / FN attentive vs blind.

## Outputs
- `faithfulness/aopc_per_image.csv`
- `faithfulness/alignment_per_image.csv`
- `faithfulness/faithfulness_metrics.json` (AOPC + NSS/CC CIs; ablation deltas if run)
- `faithfulness/aopc_curve.png`
- `faithfulness/error_taxonomy_*.csv/json`
- `faithfulness/faithfulness_status.json`

## Colab order
Mount → setup → zip → deps → **Run all** on GPU.


In [1]:
# === Colab setup ===
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

import os, sys, json, glob, shutil, zipfile
import torch

PROJECT = '/content/drive/MyDrive/DENTAL'
if not os.path.isfile(os.path.join(PROJECT, 'config.json')):
    # local / alternate mount fallback
    for cand in [os.getcwd(), os.path.abspath('..')]:
        if os.path.isfile(os.path.join(cand, 'config.json')):
            PROJECT = os.path.abspath(cand)
            break

NOTEBOOKS_DIR = os.path.join(PROJECT, 'Notebooks')
CODE_DIR = '/content/dental_code' if os.path.isdir('/content') else os.path.join(PROJECT, '_local_code')
LOCAL_DATA = '/content/dental_data' if os.path.isdir('/content') else PROJECT
LOCAL_ZIP_CACHE = '/content/dental_zip_cache' if os.path.isdir('/content') else os.path.join(PROJECT, '_zip_cache')
os.makedirs(CODE_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
os.makedirs(LOCAL_ZIP_CACHE, exist_ok=True)
os.makedirs(os.path.join(PROJECT, 'faithfulness'), exist_ok=True)

sys.path = [p for p in sys.path if p not in (CODE_DIR, PROJECT, NOTEBOOKS_DIR)]
sys.path.insert(0, CODE_DIR)
sys.path.insert(0, NOTEBOOKS_DIR)
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

with open('config.json', 'r') as f:
    CONFIG = json.load(f)

ON_GPU = torch.cuda.is_available()
device = torch.device('cuda' if ON_GPU else 'cpu')
SMOKE_TEST = False
NUM_WORKERS = 2 if ON_GPU else 0

print('PROJECT:', PROJECT)
print('LOCAL_DATA:', LOCAL_DATA, '| ON_GPU:', ON_GPU, '| device:', device)


Mounted at /content/drive
PROJECT: /content/drive/MyDrive/DENTAL
LOCAL_DATA: /content/dental_data | ON_GPU: True | device: cuda


In [2]:
# === Local images from Drive zip archives (NB04/NB05 contract) ===
import os, glob, shutil, zipfile

DATA_ZIP_DIR = os.path.join(PROJECT, 'Data zip')
DATA_MARKER = os.path.join(LOCAL_DATA, 'data', 'tdd', 'Expert', 'expert.json')


def ensure_drive_metadata():
    required = [
        'splits/test_split.csv',
        'gaze_maps_48x48',
        'core_modules.py',
        'dental_dataset.py',
        'attribution_error_taxonomy.py',
        'checkpoints/stage2_refit_best.pth',
    ]
    # refit preferred but allow best-fold fallback later
    soft = ['checkpoints/stage2_refit_best.pth']
    missing = [p for p in required if p not in soft and not os.path.exists(os.path.join(PROJECT, p))]
    if missing:
        raise FileNotFoundError('Missing on Drive: ' + ', '.join(missing))
    if not any(os.path.isfile(os.path.join(PROJECT, p)) for p in soft):
        # fold bests may still exist
        metrics = os.path.join(PROJECT, 'results', 'stage2_cv_fold_metrics.json')
        if not os.path.isfile(metrics):
            raise FileNotFoundError('Need stage2_refit_best.pth or NB04 fold metrics/ckpts')


def ensure_local_images_from_zip():
    os.makedirs(LOCAL_DATA, exist_ok=True)
    os.makedirs(LOCAL_ZIP_CACHE, exist_ok=True)
    ensure_drive_metadata()
    if os.path.exists(DATA_MARKER) or LOCAL_DATA == PROJECT:
        if os.path.exists(DATA_MARKER):
            print(f'Local images present: {os.path.join(LOCAL_DATA, "data")}')
        else:
            print(f'Using PROJECT as image root: {PROJECT}')
        return
    if not os.path.isdir(DATA_ZIP_DIR):
        raise FileNotFoundError(f'Missing zip folder: {DATA_ZIP_DIR}')
    zips = sorted(glob.glob(os.path.join(DATA_ZIP_DIR, '*.zip')))
    if not zips:
        raise FileNotFoundError(f'No .zip files in {DATA_ZIP_DIR}')
    for src_zip in zips:
        name = os.path.basename(src_zip)
        dst_zip = os.path.join(LOCAL_ZIP_CACHE, name)
        if (not os.path.exists(dst_zip)) or os.path.getsize(dst_zip) != os.path.getsize(src_zip):
            print(f'Copying zip Drive -> local cache: {name}')
            shutil.copy2(src_zip, dst_zip)
        else:
            print(f'Zip cache hit: {name}')
        print(f'Unzipping {name} -> {LOCAL_DATA} ...')
        with zipfile.ZipFile(dst_zip, 'r') as zf:
            zf.extractall(LOCAL_DATA)
    if not os.path.exists(DATA_MARKER):
        raise FileNotFoundError('Unzip finished but data/tdd/Expert/expert.json missing.')
    print(f'Local images ready: {os.path.join(LOCAL_DATA, "data")}')


ensure_local_images_from_zip()


Copying zip Drive -> local cache: data-20260715T052740Z-1-001.zip
Unzipping data-20260715T052740Z-1-001.zip -> /content/dental_data ...
Copying zip Drive -> local cache: data-20260715T052740Z-1-002.zip
Unzipping data-20260715T052740Z-1-002.zip -> /content/dental_data ...
Copying zip Drive -> local cache: data-20260715T052740Z-1-003.zip
Unzipping data-20260715T052740Z-1-003.zip -> /content/dental_data ...
Copying zip Drive -> local cache: data-20260715T052740Z-1-004.zip
Unzipping data-20260715T052740Z-1-004.zip -> /content/dental_data ...
Copying zip Drive -> local cache: data-20260715T052740Z-1-005.zip
Unzipping data-20260715T052740Z-1-005.zip -> /content/dental_data ...
Copying zip Drive -> local cache: data-20260715T052740Z-1-006.zip
Unzipping data-20260715T052740Z-1-006.zip -> /content/dental_data ...
Copying zip Drive -> local cache: data-20260715T052740Z-1-007.zip
Unzipping data-20260715T052740Z-1-007.zip -> /content/dental_data ...
Copying zip Drive -> local cache: data-20260715T

In [3]:
# === Deps (peft needs torchao>=0.16) ===
import sys, subprocess
import importlib.metadata as md
from packaging.version import Version

pkgs = [
    'torchao>=0.16.0', 'peft', 'transformers', 'scikit-learn',
    'pandas', 'numpy', 'tqdm', 'pillow', 'matplotlib', 'packaging',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
torchao_v = md.version('torchao')
print('torchao', torchao_v, '| peft', md.version('peft'))
if Version(torchao_v) < Version('0.16.0'):
    raise RuntimeError('torchao still <0.16 — Restart session, remount, re-run setup+deps')
print('Deps OK')


torchao 0.18.0 | peft 0.19.1
Deps OK


## 1 — Runtime setup & mode selection

`ON_GPU` drives real AOPC/taxonomy. CPU = preflight dry-run only.


In [4]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import pandas as pd

SEED = int(CONFIG.get('random_seed', 42))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ON_GPU = torch.cuda.is_available()
device = torch.device('cuda' if ON_GPU else 'cpu')
NUM_WORKERS = 2 if ON_GPU else 0
print('ON_GPU:', ON_GPU, '| device:', device, '| NUM_WORKERS:', NUM_WORKERS)
if ON_GPU:
    print(f'GPU: {torch.cuda.get_device_name(0)} — REAL AOPC + taxonomy will run.')
else:
    print('CPU dry-run mode — preflight only (no full AOPC/taxonomy artifacts).')


ON_GPU: True | device: cuda | NUM_WORKERS: 2
GPU: Tesla T4 — REAL AOPC + taxonomy will run.


## 2 — Local images via Drive zip cache (no copytree)

Drive keeps splits/gaze/checkpoints/results. Local holds `data/` from `Data zip/*.zip`.


In [5]:
# Idempotent zip ensure (safe after runtime restart)
if 'ensure_local_images_from_zip' in globals():
    ensure_local_images_from_zip()
else:
    print('Zip helper not in memory — re-run the zip cell above.')
print(f'Images root: {LOCAL_DATA} | cwd: {os.getcwd()}')


Local images present: /content/dental_data/data
Images root: /content/dental_data | cwd: /content/drive/MyDrive/DENTAL


## 3 — Tufts test split (AOPC + taxonomy)

Uses held-out `splits/test_split.csv` with lesion masks. On CPU, cap to a few samples for preflight.


In [ ]:
# Refresh modules from Drive
import shutil
for fname in (
    'core_modules.py', 'dental_dataset.py', 'attribution_error_taxonomy.py',
    'stage2_train_utils.py', 'faithfulness_metrics.py', 'ood_hardening.py',
):
    src = os.path.join(PROJECT, fname)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(CODE_DIR, fname))
for mod in (
    'core_modules', 'dental_dataset', 'attribution_error_taxonomy',
    'stage2_train_utils', 'faithfulness_metrics',
):
    if mod in sys.modules:
        del sys.modules[mod]

from dental_dataset import DentalDataset

test_csv = os.path.join(PROJECT, 'splits', 'test_split.csv')
test_ds = DentalDataset(test_csv, PROJECT, image_root=LOCAL_DATA, augment=False)
# FORCE_FULL_FAITHFULNESS=1 runs the full test set even on CPU (slow but paper-ready).
FORCE_FULL = os.environ.get('FORCE_FULL_FAITHFULNESS', '0') == '1'
if (not ON_GPU) and (not FORCE_FULL):
    test_ds.df = test_ds.df.head(8).reset_index(drop=True)
    print(f'CPU dry-run: capped Tufts test to {len(test_ds)} images')
elif not ON_GPU and FORCE_FULL:
    print(f'CPU FULL faithfulness pass: {len(test_ds)} images (slow)')

test_loader = DataLoader(
    test_ds,
    batch_size=min(8 if not ON_GPU else int(CONFIG.get('chosen_batch_size', 64)), max(1, len(test_ds))),
    shuffle=False,
    num_workers=0 if not ON_GPU else NUM_WORKERS,
    pin_memory=ON_GPU,
)
print(f'Tufts test loader: {len(test_ds)} images | batches: {len(test_loader)}')


## 4 — Model + helpers

Load refit (gaze) weights; optionally best λ=0 ablation fold for alignment/AOPC contrast.


In [ ]:
import torch.nn.functional as F
from core_modules import SwinGazeModel, calibrate_decision_threshold
from faithfulness_metrics import (
    per_image_aopc_and_alignment,
    bootstrap_mean_ci,
    bootstrap_paired_diff_ci,
    aopc_auc_from_drops,
    mask_top_k_attention,
)
import importlib.metadata as md
from packaging.version import Version

torchao_v = md.version('torchao')
if Version(torchao_v) < Version('0.16.0'):
    raise RuntimeError(f'torchao {torchao_v} < 0.16 — run deps cell + Restart session')


def resolve_stage2_ckpt(project='.'):
    candidates = [
        os.path.join(project, 'checkpoints', 'stage2_refit_best.pth'),
        os.path.join(project, 'checkpoints', 'stage2_best.pth'),
    ]
    metrics_path = os.path.join(project, 'results', 'stage2_cv_fold_metrics.json')
    if os.path.isfile(metrics_path):
        with open(metrics_path, 'r', encoding='utf-8') as f:
            rows = json.load(f)
        if rows:
            best = max(rows, key=lambda r: float(r.get('val_AUROC', -1)))
            ck = best.get('checkpoint') or f"checkpoints/stage2_gaze_fold{best['fold']}_best.pth"
            candidates.insert(1, ck if os.path.isabs(ck) else os.path.join(project, ck))
    for p in candidates:
        p = os.path.normpath(p)
        if os.path.isfile(p):
            return p
    raise FileNotFoundError('No Stage-2 checkpoint (need refit or NB04 fold best).')


def resolve_ablation_ckpt(project='.'):
    """Best λ=0 fold by val_AUROC, else any ablation best file."""
    metrics_path = os.path.join(project, 'results', 'stage2_part1_negative_report.json')
    if os.path.isfile(metrics_path):
        with open(metrics_path, 'r', encoding='utf-8') as f:
            rep = json.load(f)
        rows = rep.get('cv_ablation_fold_metrics') or []
        if rows:
            best = max(rows, key=lambda r: float(r.get('val_AUROC', -1)))
            ck = best.get('checkpoint') or f"checkpoints/stage2_ablation_fold{best['fold']}_best.pth"
            p = ck if os.path.isabs(ck) else os.path.join(project, ck)
            if os.path.isfile(p):
                return os.path.normpath(p), best
    # fallback: highest fold index present
    for fold in range(5):
        p = os.path.join(project, 'checkpoints', f'stage2_ablation_fold{fold}_best.pth')
        if os.path.isfile(p):
            return os.path.normpath(p), {'fold': fold}
    return None, None


def load_model(ckpt_path):
    m = SwinGazeModel(CONFIG).to(device)
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    state = state['model'] if isinstance(state, dict) and 'model' in state else state
    m.load_state_dict(state, strict=False)
    m.eval()
    return m


def extract_last_attn(out):
    last = out.attentions[-1]
    return last.mean(dim=1).mean(dim=1).reshape(-1, 7, 7)


STAGE2_CKPT = resolve_stage2_ckpt(PROJECT)
print('Loading gaze/refit:', STAGE2_CKPT)
model = load_model(STAGE2_CKPT)

ABLATION_CKPT, ABLATION_META = resolve_ablation_ckpt(PROJECT)
model_ablation = None
if ABLATION_CKPT:
    print('Loading ablation:', ABLATION_CKPT, ABLATION_META)
    model_ablation = load_model(ABLATION_CKPT)
else:
    print('No ablation checkpoint found — NSS/CC vs ablation skipped.')

K_LEVELS = [0.05, 0.10, 0.15, 0.20, 0.50]
K_COLS = [f'k_{int(k*100)}pct' for k in K_LEVELS]
N_BOOT = int(CONFIG.get('negative_result_bootstrap_n', 2000))
print('Model(s) ready. N_BOOT=', N_BOOT)


## 5 — Pre-flight (always runs)

One-batch AOPC + NSS/CC path check + taxonomy smoke.


In [ ]:
from attribution_error_taxonomy import classify_error, summarize_taxonomy

CODE_CHECK_PASSED = False
model.eval()
batch = next(iter(test_loader))
out = per_image_aopc_and_alignment(
    model, batch, device, k_levels=K_LEVELS, extract_last_attn=extract_last_attn
)
assert np.isfinite(out['aopc_auc']).all()
assert out['nss'] is not None
# taxonomy smoke
images = batch['image'].to(device)
with torch.no_grad():
    logits, _, last_attn, hout = model(images)
    probs = torch.sigmoid(logits.squeeze(-1))
    attn = last_attn if last_attn is not None else extract_last_attn(hout)
y0 = int(batch['has_lesion_gt'][0].item())
pred0 = int(float(probs[0].item()) > 0.5)
mask0 = batch['bbox_mask'][0].detach().cpu().numpy()
if mask0.ndim == 3:
    mask0 = mask0[0]
a0 = attn[0].detach().cpu().numpy()
a0_u = F.interpolate(
    torch.from_numpy(a0)[None, None].float(), size=mask0.shape[-2:], mode='bilinear', align_corners=False
).squeeze().numpy()
rec = classify_error(y0, pred0, a0_u, mask0)
assert 'bucket' in rec
n_gaze = int(np.nansum(np.isfinite(out['nss']) & out['has_gaze']))
print(
    f"Preflight OK | batch={len(out['aopc_auc'])} | mean AOPC={float(np.mean(out['aopc_auc'])):.4f} "
    f"| NSS finite={n_gaze} | tax={rec['bucket']}"
)
CODE_CHECK_PASSED = True


## 6 — Per-image AOPC + NSS/CC (GPU, or CPU if FORCE_FULL_FAITHFULNESS=1)

Primary: bootstrap CI over images for AOPC_AUC, NSS, CC. Optional ablation contrast.


In [ ]:
REAL_AOPC_DONE = False
RUN_FULL = ON_GPU or (os.environ.get('FORCE_FULL_FAITHFULNESS', '0') == '1')
aopc_df = None
align_df = None
aopc_auc = None
aopc_score = None
faithfulness_report = None

def _eval_loader(net, loader, tag):
    rows_aopc, rows_align = [], []
    net.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'{tag}', unit='batch'):
            out = per_image_aopc_and_alignment(
                net, batch, device, k_levels=K_LEVELS, extract_last_attn=extract_last_attn
            )
            B = len(out['aopc_auc'])
            ids = out['ids'] if out['ids'] is not None else list(range(len(rows_aopc), len(rows_aopc) + B))
            for i in range(B):
                r = {'id': ids[i] if i < len(ids) else i, 'aopc_auc': float(out['aopc_auc'][i])}
                for k in K_LEVELS:
                    r[f'k_{int(k*100)}pct'] = float(out['drops'][f'k_{int(k*100)}pct'][i])
                rows_aopc.append(r)
                rows_align.append({
                    'id': r['id'],
                    'has_gaze': bool(out['has_gaze'][i]),
                    'nss': None if not np.isfinite(out['nss'][i]) else float(out['nss'][i]),
                    'cc': None if not np.isfinite(out['cc'][i]) else float(out['cc'][i]),
                })
    return pd.DataFrame(rows_aopc), pd.DataFrame(rows_align)


if not RUN_FULL:
    print('SKIPPING full faithfulness — CPU dry-run (preflight only). Set FORCE_FULL_FAITHFULNESS=1 to override.')
else:
    os.makedirs('faithfulness', exist_ok=True)
    aopc_df, align_df = _eval_loader(model, test_loader, 'gaze/refit')
    aopc_df.to_csv('faithfulness/aopc_per_image.csv', index=False)
    align_df.to_csv('faithfulness/alignment_per_image.csv', index=False)

    # mean curve over images (for plot)
    mean_drops = aopc_df[K_COLS].mean(axis=0).values
    aopc_auc = float(aopc_auc_from_drops(K_LEVELS, mean_drops))
    aopc_score = float(mean_drops.mean())
    pd.DataFrame([dict(zip(K_COLS, mean_drops))]).to_csv('faithfulness/aopc_curves.csv', index=False)

    aopc_ci = bootstrap_mean_ci(aopc_df['aopc_auc'].tolist(), n_boot=N_BOOT)
    nss_vals = [v for v in align_df.loc[align_df['has_gaze'], 'nss'].tolist() if v is not None]
    cc_vals = [v for v in align_df.loc[align_df['has_gaze'], 'cc'].tolist() if v is not None]
    nss_ci = bootstrap_mean_ci(nss_vals, n_boot=N_BOOT, seed=43)
    cc_ci = bootstrap_mean_ci(cc_vals, n_boot=N_BOOT, seed=44)

    faithfulness_report = {
        'checkpoint': STAGE2_CKPT,
        'n_test_images': int(len(aopc_df)),
        'n_with_gaze': int(align_df['has_gaze'].sum()),
        'k_levels': K_LEVELS,
        'AOPC_AUC_curve_mean': aopc_auc,
        'AOPC_naive_mean_drop': aopc_score,
        'AOPC_AUC_per_image_bootstrap': aopc_ci,
        'NSS_bootstrap': nss_ci,
        'CC_bootstrap': cc_ci,
        'mean_confidence_drops': mean_drops.tolist(),
        'ablation': None,
    }
    print(f"AOPC_AUC (curve-mean)={aopc_auc:.4f} | per-image mean={aopc_ci['mean']:.4f} "
          f"[{aopc_ci['lower']:.4f}, {aopc_ci['upper']:.4f}]")
    print(f"NSS={nss_ci['mean']:.4f} [{nss_ci['lower']:.4f}, {nss_ci['upper']:.4f}] | "
          f"CC={cc_ci['mean']:.4f} [{cc_ci['lower']:.4f}, {cc_ci['upper']:.4f}] "
          f"(n_gaze={nss_ci['n']})")

    if model_ablation is not None:
        aopc_ab, align_ab = _eval_loader(model_ablation, test_loader, 'ablation')
        aopc_ab.to_csv('faithfulness/aopc_per_image_ablation.csv', index=False)
        align_ab.to_csv('faithfulness/alignment_per_image_ablation.csv', index=False)
        # align on id
        m = aopc_df.merge(aopc_ab, on='id', suffixes=('_gaze', '_ablation'))
        am = align_df.merge(align_ab, on='id', suffixes=('_gaze', '_ablation'))
        am = am[am['has_gaze_gaze'] & am['has_gaze_ablation']]
        d_aopc = bootstrap_paired_diff_ci(m['aopc_auc_ablation'], m['aopc_auc_gaze'], n_boot=N_BOOT, seed=45)
        d_nss = bootstrap_paired_diff_ci(
            [x for x in am['nss_ablation'].tolist() if x is not None],
            [x for x in am['nss_gaze'].tolist() if x is not None],
            n_boot=N_BOOT, seed=46,
        )
        d_cc = bootstrap_paired_diff_ci(
            [x for x in am['cc_ablation'].tolist() if x is not None],
            [x for x in am['cc_gaze'].tolist() if x is not None],
            n_boot=N_BOOT, seed=47,
        )
        faithfulness_report['ablation'] = {
            'checkpoint': ABLATION_CKPT,
            'meta': ABLATION_META,
            'AOPC_AUC_per_image_bootstrap': bootstrap_mean_ci(aopc_ab['aopc_auc'].tolist(), n_boot=N_BOOT, seed=48),
            'NSS_bootstrap': bootstrap_mean_ci(
                [v for v in align_ab.loc[align_ab['has_gaze'], 'nss'].tolist() if v is not None],
                n_boot=N_BOOT, seed=49,
            ),
            'CC_bootstrap': bootstrap_mean_ci(
                [v for v in align_ab.loc[align_ab['has_gaze'], 'cc'].tolist() if v is not None],
                n_boot=N_BOOT, seed=50,
            ),
            'paired_diff_gaze_minus_ablation': {
                'AOPC_AUC': d_aopc,
                'NSS': d_nss,
                'CC': d_cc,
            },
            'note': 'Positive diff => gaze model higher than λ=0 ablation on that metric.',
        }
        print('Ablation contrast (gaze - ablation):')
        print(f"  dAOPC={d_aopc['mean_diff']:+.4f} [{d_aopc['lower']:+.4f}, {d_aopc['upper']:+.4f}]")
        print(f"  dNSS ={d_nss['mean_diff']:+.4f} [{d_nss['lower']:+.4f}, {d_nss['upper']:+.4f}]")
        print(f"  dCC  ={d_cc['mean_diff']:+.4f} [{d_cc['lower']:+.4f}, {d_cc['upper']:+.4f}]")

    with open('faithfulness/faithfulness_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(faithfulness_report, f, indent=2)
    print('Saved faithfulness/faithfulness_metrics.json')
    REAL_AOPC_DONE = True


## 6b — Error-attribution taxonomy (GPU only)

FP Type A (hallucination), FP Type B adapted (right label / wrong place),  
FN attentive vs blind. Uses last-attn + `bbox_mask`.


In [10]:
REAL_TAXONOMY_DONE = False
taxonomy_summary = None

RUN_FULL = ON_GPU or (os.environ.get('FORCE_FULL_FAITHFULNESS', '0') == '1')
if not RUN_FULL:
    print('SKIPPING full taxonomy — CPU dry-run.')
else:
    from attribution_error_taxonomy import classify_error, summarize_taxonomy

    model.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for batch in test_loader:
            logits, _, _, _ = model(batch['image'].to(device))
            probs = torch.sigmoid(logits.squeeze(-1)).cpu().numpy()
            all_y.extend(batch['has_lesion_gt'].cpu().numpy().tolist())
            all_p.extend(probs.tolist())
    thr = float(calibrate_decision_threshold(np.asarray(all_y), np.asarray(all_p)))
    print(f'Taxonomy threshold (Youden): {thr:.4f}')

    rows = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Taxonomy', unit='batch'):
            images = batch['image'].to(device)
            logits, _, last_attn, out = model(images)
            probs = torch.sigmoid(logits.squeeze(-1)).cpu().numpy()
            preds = (probs > thr).astype(int)
            ys = batch['has_lesion_gt'].cpu().numpy().astype(int)
            masks = batch['bbox_mask'].cpu().numpy()
            attn = (last_attn if last_attn is not None else extract_last_attn(out)).cpu().numpy()
            ids = batch.get('image_id', batch.get('id', list(range(len(ys)))))
            if torch.is_tensor(ids):
                ids = ids.tolist()
            for i in range(len(ys)):
                mask_i = masks[i][0] if masks[i].ndim == 3 else masks[i]
                a = attn[i]
                if a.shape != mask_i.shape:
                    a = F.interpolate(
                        torch.from_numpy(a)[None, None].float(),
                        size=mask_i.shape[-2:], mode='bilinear', align_corners=False,
                    ).squeeze().numpy()
                rec = classify_error(int(ys[i]), int(preds[i]), a, mask_i, tau=0.10, type_b_top_frac=0.30)
                rec['id'] = ids[i] if i < len(ids) else i
                rec['prob'] = float(probs[i])
                rec['threshold'] = thr
                rows.append(rec)

    tax_df = pd.DataFrame(rows)
    taxonomy_summary = summarize_taxonomy(rows)
    taxonomy_summary['checkpoint'] = STAGE2_CKPT
    taxonomy_summary['threshold'] = thr
    os.makedirs('faithfulness', exist_ok=True)
    tax_df.to_csv('faithfulness/error_taxonomy_per_image.csv', index=False)
    with open('faithfulness/error_taxonomy_summary.json', 'w') as f:
        json.dump(taxonomy_summary, f, indent=2)
    print('Taxonomy counts:', taxonomy_summary.get('counts'))
    REAL_TAXONOMY_DONE = True


Taxonomy threshold (Youden): 0.8191


Taxonomy:   0%|          | 0/2 [00:00<?, ?batch/s]

Taxonomy counts: {'tn': 56, 'fp_type_b_misattribution': 8, 'fn_blind': 25, 'fp_type_a_hallucination': 10, 'fn_attentive': 1}


## 7 — Save run status


In [ ]:
status = {
    'on_gpu': bool(ON_GPU),
    'force_full': os.environ.get('FORCE_FULL_FAITHFULNESS', '0') == '1',
    'code_check_passed': bool(globals().get('CODE_CHECK_PASSED', False)),
    'real_aopc_done': bool(globals().get('REAL_AOPC_DONE', False)),
    'real_taxonomy_done': bool(globals().get('REAL_TAXONOMY_DONE', False)),
    'checkpoint': globals().get('STAGE2_CKPT'),
    'ablation_checkpoint': globals().get('ABLATION_CKPT'),
    'aopc_auc_curve_mean': None if aopc_auc is None else float(aopc_auc),
    'aopc_per_image_ci': None if not faithfulness_report else faithfulness_report.get('AOPC_AUC_per_image_bootstrap'),
    'nss_ci': None if not faithfulness_report else faithfulness_report.get('NSS_bootstrap'),
    'cc_ci': None if not faithfulness_report else faithfulness_report.get('CC_bootstrap'),
    'n_test_images': len(test_loader.dataset) if 'test_loader' in globals() else None,
    'taxonomy_counts': None if not taxonomy_summary else taxonomy_summary.get('counts'),
}
os.makedirs('faithfulness', exist_ok=True)
with open('faithfulness/faithfulness_status.json', 'w', encoding='utf-8') as f:
    json.dump(status, f, indent=2)
print(json.dumps(status, indent=2))


## 8 — AOPC curve plot (GPU runs only)


In [ ]:
if REAL_AOPC_DONE and aopc_df is not None:
    import matplotlib.pyplot as plt
    means = aopc_df[K_COLS].mean(axis=0).values
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(K_LEVELS, means, marker='o', lw=2)
    ax.set_xlabel('Top-k fraction masked')
    ax.set_ylabel('Mean confidence drop')
    ci = faithfulness_report['AOPC_AUC_per_image_bootstrap']
    ax.set_title(f"AOPC curve (per-image mean={ci['mean']:.4f} [{ci['lower']:.4f},{ci['upper']:.4f}])")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('faithfulness/aopc_curve.png', dpi=150)
    plt.show()
    print('Saved faithfulness/aopc_curve.png')
else:
    print('No AOPC plot — dry-run or faithfulness not completed.')


## 9 — Final status banner


In [ ]:
from IPython.display import Markdown, display

if REAL_AOPC_DONE and faithfulness_report is not None:
    ci = faithfulness_report['AOPC_AUC_per_image_bootstrap']
    nss = faithfulness_report['NSS_bootstrap']
    cc = faithfulness_report['CC_bootstrap']
    ab = faithfulness_report.get('ablation')
    ab_line = 'not run'
    if ab and ab.get('paired_diff_gaze_minus_ablation'):
        d = ab['paired_diff_gaze_minus_ablation']
        ab_line = (
            f"dAOPC={d['AOPC_AUC']['mean_diff']:+.4f}, "
            f"dNSS={d['NSS']['mean_diff']:+.4f}, "
            f"dCC={d['CC']['mean_diff']:+.4f}"
        )
    msg = f"""
### NB06 COMPLETE — Faithfulness hardened
- Checkpoint: `{STAGE2_CKPT}`
- **AOPC_AUC (per-image):** **{ci['mean']:.4f}** [{ci['lower']:.4f}, {ci['upper']:.4f}]
- **NSS:** {nss['mean']:.4f} [{nss['lower']:.4f}, {nss['upper']:.4f}]
- **CC:** {cc['mean']:.4f} [{cc['lower']:.4f}, {cc['upper']:.4f}]
- Gaze vs ablation: {ab_line}
- Taxonomy: {'done' if REAL_TAXONOMY_DONE else 'not run'}
"""
else:
    msg = f"""
### NB06 CPU DRY-RUN
- Preflight: {'PASSED' if globals().get('CODE_CHECK_PASSED') else 'NOT RUN'}
- Full pass skipped. On Colab GPU: Runtime → Run all.
- Or locally: `FORCE_FULL_FAITHFULNESS=1 python run_faithfulness_hardening.py`
"""
display(Markdown(msg))
